In [ ]:
from datasets import load_dataset
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# ---------------------------------------------------------
# 1. DATA PREP & SPLITTING
# ---------------------------------------------------------
# Load dataset (Replace with your specific dataset)
dataset = load_dataset("json", data_files="your_dataset.jsonl", split="train")

def prep_data(ds):
    ds['label'] = ds['rating'] / 5.0  # Normalize to 0-1 for regression
    return ds

columns_to_drop = [col for col in dataset.column_names if col not in ['text', 'label']]
hf_dataset = dataset.map(prep_data, num_proc=8, remove_columns=columns_to_drop)

# Tokenize
tokenizer = RobertaTokenizer.from_pretrained('FacebookAI/roberta-base')
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=128)

tokenized_dataset = hf_dataset.map(tokenize, batched=True, num_proc=8)

# The 80/10/10 Split
first_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_data = first_split["train"]
holdout_data = first_split["test"]

second_split = holdout_data.train_test_split(test_size=0.5, seed=42)
val_data = second_split["train"]
test_data = second_split["test"]

# Format for PyTorch
columns_to_keep = ['input_ids', 'attention_mask', 'label']
train_data.set_format(type='torch', columns=columns_to_keep)
val_data.set_format(type='torch', columns=columns_to_keep)
test_data.set_format(type='torch', columns=columns_to_keep)

# ---------------------------------------------------------
# 2. MODEL INITIALIZATION & TRAINING (Jetson/128GB Optimized)
# ---------------------------------------------------------
model = RobertaForSequenceClassification.from_pretrained('FacebookAI/roberta-base', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

training_args = TrainingArguments(
    output_dir="./roberta_sentiment_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=5e-5,
    per_device_train_batch_size=64, # Optimized for high RAM
    per_device_eval_batch_size=64,
    num_train_epochs=10,            # High epoch limit, protected by Early Stopping
    weight_decay=0.01,
    bf16=True,                      # High precision/speed math
    save_total_limit=2,
    dataloader_num_workers=8        # CPU parallel processing
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting training...")
trainer.train()

# ---------------------------------------------------------
# 3. EVALUATION & VISUALIZATION (Pure Matplotlib)
# ---------------------------------------------------------
# Extract loss logs
logs = trainer.state.log_history
train_steps = [log['step'] for log in logs if 'loss' in log]
train_loss = [log['loss'] for log in logs if 'loss' in log]
eval_steps = [log['step'] for log in logs if 'eval_loss' in log]
eval_loss = [log['eval_loss'] for log in logs if 'eval_loss' in log]

# Plot Learning Curve
plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_loss, label='Training Loss', color='#1f77b4', linewidth=2)
plt.plot(eval_steps, eval_loss, label='Validation Loss', color='#ff7f0e', linewidth=2, marker='o')
plt.grid(True, linestyle='--', alpha=0.7)
plt.title('Model Learning Curve: Train vs. Validation Loss', fontsize=14, fontweight='bold')
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.legend(fontsize=12)
plt.show()

# Calculate Final Test MSE
pred = trainer.predict(test_data)
y_pred = pred.predictions.flatten()
y_true = pred.label_ids
mse = np.mean((y_true - y_pred) ** 2)
print(f"Final Test Mean Squared Error: {mse:.4f}")